# cfd10 — Colab teacher training

**Before running:** set the runtime to a GPU (Runtime -> Change runtime type -> **T4** or **L4**).

Then run the cells top to bottom. The only thing you must edit is `GDRIVE_DATA` in cell 4 — point it at the Drive folder that holds the `raw_v16` CSVs (the same data that is local in `data/raw_v16`).

In [ ]:
# Cell 1 - Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 - Clone the (public) repo and install deps
import os
from pathlib import Path
REPO_DIR = Path('/content/cfd10')
REPO_URL = 'https://github.com/Sovenski/cfd10.git'
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull --ff-only -q
# Installs core deps from pyproject; does NOT touch Colab's GPU torch (torch is a dev-only extra).
!pip install -q -e .
!git rev-parse --short HEAD

In [ ]:
# Cell 3 - Link the Drive data into the repo + GPU check
import os, torch
GDRIVE_DATA = '/content/drive/MyDrive/cfd10/data/raw_v16'   # <-- EDIT to your raw_v16 folder in Drive
os.makedirs('/content/cfd10/data', exist_ok=True)
link = '/content/cfd10/data/raw_v16'
if not os.path.exists(link):
    os.symlink(GDRIVE_DATA, link)
assert os.path.isdir(link), f'data not found: {link} -> {GDRIVE_DATA}'
n_csv = len([f for f in os.listdir(link) if f.endswith('.csv')])
print('CSV files visible:', n_csv)
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU-only')

In [ ]:
# Cell 4 - Train the TCN teacher (the beat-the-GBDT-baseline gate). ~minutes on a T4/L4.
!python pipeline/fit_teacher.py --epochs 30

In [ ]:
# Cell 5 - Show the teacher scorecard (TCN vs GBDT baseline, per side, pooled + SPX)
print(open('/content/cfd10/outputs/teacher_pooled_scorecard.md', encoding='utf-8').read())

In [ ]:
# Cell 6 (optional) - GBDT pooled baseline + label QA for reference
!python pipeline/fit_pooled.py
print(open('/content/cfd10/outputs/baseline_pooled_scorecard.md', encoding='utf-8').read())

## After running
Copy the scorecards back so we can decide the next step:
```python
import shutil, os
os.makedirs('/content/drive/MyDrive/cfd10/outputs', exist_ok=True)
for f in ['teacher_pooled_scorecard.md', 'baseline_pooled_scorecard.md']:
    p = f'/content/cfd10/outputs/{f}'
    if os.path.exists(p):
        shutil.copy(p, f'/content/drive/MyDrive/cfd10/outputs/{f}')
```
Then paste the teacher scorecard back here. If the TCN beats the GBDT baseline on the LOW side out-of-sample, we proceed to distillation (Phase 8).